# Single-GPU Training: OLMo Earth Firescar Fine-tuning

Interactive training notebook with live loss curves and metric tracking.

In [ ]:
import json
import os
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr
from IPython.display import clear_output
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.abspath(".."))
from firescars.checkpoint import save_checkpoint
from firescars.dataset import FirescarDataset
from firescars.evaluate import evaluate
from firescars.loss import FirescarLoss
from firescars.model import FirescarModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Create Synthetic Dataset (for demo)

Replace this cell with real data loading when chips are available.

In [ ]:
DATA_PATH = "/tmp/firescar_chips"
CKPT_PATH = "/tmp/firescar_checkpoints/"

# Create synthetic data if not present
if not os.path.exists(os.path.join(DATA_PATH, "train")):
    os.makedirs(DATA_PATH, exist_ok=True)
    store = zarr.open(DATA_PATH, mode="w")
    for split, n in [("train", 64), ("val", 16), ("test", 16)]:
        grp = store.create_group(split)
        # Simulate burned areas with spatial structure
        imgs = np.random.randint(500, 3000, (n, 6, 224, 224), dtype=np.int16)
        masks = np.zeros((n, 224, 224), dtype=np.uint8)
        for i in range(n):
            # Random burn patches
            cx, cy = np.random.randint(40, 184, 2)
            r = np.random.randint(20, 60)
            yy, xx = np.ogrid[:224, :224]
            burn = ((xx - cx) ** 2 + (yy - cy) ** 2) < r**2
            masks[i] = burn.astype(np.uint8)
            # Darken SWIR bands in burned area (simulate burn signal)
            imgs[i, 4:, burn] = imgs[i, 4:, burn] // 3
        grp.create_array("imagery", data=imgs)
        grp.create_array("masks", data=masks)

    with open(os.path.join(DATA_PATH, "metadata.json"), "w") as f:
        json.dump({"band_stats": None}, f)
    print("Synthetic dataset created")
else:
    print("Dataset already exists")

# Clear old checkpoints for fresh training
if os.path.exists(CKPT_PATH):
    shutil.rmtree(CKPT_PATH)
print("Ready to train")

## 2. Setup Model and Training

In [ ]:
# Hyperparameters
EPOCHS = 20
BATCH_SIZE = 8
LR = 1e-4
MIN_LR = 1e-6
WARMUP_STEPS = 20
ENCODER_LR_MULT = 0.1
PATIENCE = 7

# Model
model = FirescarModel(
    encoder_name="vit_base_patch16_224",
    in_chans=6,
    img_size=224,
    pretrained_encoder=True,
).to(device)

# Data
train_ds = FirescarDataset(DATA_PATH, split="train", augment=True)
val_ds = FirescarDataset(DATA_PATH, split="val", augment=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

# Optimizer with encoder LR multiplier
param_groups = model.get_param_groups(ENCODER_LR_MULT)
optimizer = AdamW(
    [
        {"params": param_groups[0]["params"], "lr": LR * ENCODER_LR_MULT},
        {"params": param_groups[1]["params"], "lr": LR},
    ],
    weight_decay=0.01,
)

# Scheduler
total_steps = EPOCHS * len(train_loader)
warmup = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_STEPS)
cosine = CosineAnnealingLR(optimizer, T_max=total_steps - WARMUP_STEPS, eta_min=MIN_LR)
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[WARMUP_STEPS])

criterion = FirescarLoss(bce_weight=0.5, dice_weight=0.5)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Train: {len(train_ds)} chips, Val: {len(val_ds)} chips")
print(f"Steps/epoch: {len(train_loader)}, Total steps: {total_steps}")

## 3. Training Loop with Live Plots

In [ ]:
# Training history
history = {"train_loss": [], "val_loss": [], "iou": [], "f1": [], "lr": []}
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(EPOCHS):
    # Train
    model.train()
    epoch_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        loss = criterion(logits, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item() * imgs.size(0)

    train_loss = epoch_loss / len(train_ds)

    # Validate
    val_metrics = evaluate(model, val_loader, criterion, device)

    # Record
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["iou"].append(val_metrics["iou"])
    history["f1"].append(val_metrics["f1"])
    history["lr"].append(optimizer.param_groups[1]["lr"])

    # Checkpoint
    is_best = val_metrics["loss"] < best_val_loss
    if is_best:
        best_val_loss = val_metrics["loss"]
        patience_counter = 0
    else:
        patience_counter += 1

    save_checkpoint(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "val_metrics": val_metrics,
            "history": history,
        },
        CKPT_PATH,
        is_best=is_best,
    )

    # Live plot
    clear_output(wait=True)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Loss curves
    axes[0].plot(history["train_loss"], "b-", label="Train")
    axes[0].plot(history["val_loss"], "r-", label="Val")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss Curves")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Metrics
    axes[1].plot(history["iou"], "g-", label="IoU")
    axes[1].plot(history["f1"], "m-", label="F1")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_title("Validation Metrics")
    axes[1].legend()
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)

    # Learning rate
    axes[2].plot(history["lr"], "k-")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("LR")
    axes[2].set_title("Learning Rate Schedule")
    axes[2].set_yscale("log")
    axes[2].grid(True, alpha=0.3)

    plt.suptitle(
        f"Epoch {epoch + 1}/{EPOCHS} | Val Loss: {val_metrics['loss']:.4f} | "
        f"IoU: {val_metrics['iou']:.4f} | F1: {val_metrics['f1']:.4f}",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch + 1}")
        break

print(f"\nTraining complete. Best val_loss: {best_val_loss:.4f}")

## 4. Final Training Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

epochs_range = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_range, history["train_loss"], "b-o", markersize=4, label="Train Loss")
axes[0].plot(epochs_range, history["val_loss"], "r-o", markersize=4, label="Val Loss")
axes[0].axhline(
    y=best_val_loss, color="r", linestyle="--", alpha=0.5, label=f"Best: {best_val_loss:.4f}"
)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (BCE + Dice)")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history["iou"], "g-o", markersize=4, label="IoU")
axes[1].plot(epochs_range, history["f1"], "m-o", markersize=4, label="F1")
axes[1].axhline(y=0.75, color="g", linestyle="--", alpha=0.5, label="IoU target (0.75)")
axes[1].axhline(y=0.80, color="m", linestyle="--", alpha=0.5, label="F1 target (0.80)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_title("Validation Metrics")
axes[1].legend()
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")